# HW2 — Classifying Rice Variety with Logistic Regression
**FRA503 Introduction to Machine Learning (1/2569)**

*Phattanarat Jeedjeen 68340700410*

---


## Table of contents
- [1. Problem statement](#1--problem-statement)
- [2. Data loading & quality audit](#2--data-loading--quality-audit)
- [3. Train / test split](#3--train--test-split)
- [4. Target distribution](#4--target-distribution)
- [5. Initial feature exploration](#5--initial-feature-exploration)
- [6. Outlier analysis](#6--outlier-analysis)
- [7. Missing-value analysis](#7--missing-value-analysis)
- [8. Feature selection — continuous features](#8--feature-selection--continuous-features)
- [9. Feature selection — categorical features](#9--feature-selection--categorical-features)
- [10. Redundant-feature analysis](#10--redundant-feature-analysis)
- [11. Feature extraction](#11--feature-extraction)
- [12. Data pre-processing for ML](#12--data-pre-processing-for-ml)
- [13. Model building & training](#13--model-building-and-training)
- [14. Results and presentation](#14--results-and-presentation)
- [15. Discussion](#15--discussion)
- [16. Conclusion](#16--conclusion)

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEED = 42
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60); pd.set_option('display.max_rows', 100)
def _fmt(v):
    if v != v: return 'nan'
    if v == 0: return '0'
    if abs(v) < 1e-4: return f'{v:.2e}'
    if abs(v) >= 1e5: return f'{v:,.0f}'
    return f'{v:,.4f}'.rstrip('0').rstrip('.') if abs(v) < 1000 else f'{v:,.2f}'.rstrip('0').rstrip('.')
pd.set_option('display.float_format', _fmt)

BLUE, ORANGE, AQUA, YELLOW, MAGENTA, GREY, INK = '#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#8a8a85', '#0b0b0b'
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e4e3de', 'grid.linewidth': 0.6, 'axes.axisbelow': True,
    'axes.prop_cycle': plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, MAGENTA]),
    'axes.titlesize': 10, 'axes.labelsize': 9, 'xtick.labelsize': 8, 'ytick.labelsize': 8,
    'legend.frameon': False, 'legend.fontsize': 8, 'figure.autolayout': False})
print('setup ok  | numpy', np.__version__, '| pandas', pd.__version__)

# 1 · Problem statement

### 1.1 Data description
The **Paddy dataset** (UCI ML Repository, DOI 10.24432/C55W3J, CC BY 4.0) holds rice-cultivation records from six agri-blocks in Tamil Nadu, India (Cuddalore, Kurinjipadi, Panruti, Kallakurichi, Chinnasalem, Sankarapuram). Each record covers the plot (size, block, soil type), the rice **variety** planted, nursery and land preparation, the amounts of fertiliser, herbicide and pesticide applied on given days, the weather in each growth window (rainfall and an "AI" variable, min/max temperature, wind speed/direction, relative humidity), and two harvest measurements: `Trash` (bundles) and `Paddy yield` (kg). UCI reports 2,790 instances × 45 columns with *no missing values*.

## Variable description
**Role** column: **T** = target, **P** = candidate predictor.

| # | Feature | Role | Type | Definition |
|---|---|---|---|---|
| 1 | `Hectares` | P | Numerical (Integer) | Plot size, in hectares |
| 2 | `Agriblock` | P | Categorical | Agricultural block / district where the plot is located |
| 3 | **`Variety`** | **T** | **Categorical (3 classes)** | **Rice variety planted: `CO_43`, `ponmani`, `delux ponni`** |
| 4 | `Soil Types` | P | Categorical | Soil type, e.g. alluvial or clay |
| 5 | `Seedrate (in Kg)` | P | Numerical (Integer) | Seed quantity used, in kg |
| 6 | `LP_Mainfield (in Tonnes)` | P | Numerical (Continuous) | Land-preparation material/manure used in the main field, in tonnes; exact material not specified |
| 7 | `Nursery` | P | Categorical | Nursery preparation method, e.g. dry or wet |
| 8 | `Nursery area (Cents)` | P | Numerical (Integer) | Nursery area, in cents |
| 9 | `LP_nurseryarea (in Tonnes)` | P | Numerical (Integer) | Manure used to prepare the nursery area, in tonnes |
| 10 | `DAP_20days` | P | Numerical (Integer) | DAP fertiliser applied around day 20 |
| 11 | `Weed28D_thiobencarb` | P | Numerical | Thiobencarb herbicide applied around day 28 |
| 12 | `Urea_40Days` | P | Numerical (Continuous) | Urea fertiliser applied around day 40 |
| 13 | `Potassh_50Days` | P | Numerical (Continuous) | Potash/potassium fertiliser applied around day 50 |
| 14 | `Micronutrients_70Days` | P | Numerical | Micronutrients applied around day 70 |
| 15 | `Pest_60Day (in ml)` | P | Numerical | Pesticide applied around day 60, in ml |
| 16 | `30DRain (in mm)` | P | Numerical (Continuous) | Rainfall in roughly the first 30 days, in mm |
| 17 | `30DAI (in mm)` | P | Numerical (Continuous) | "AI" variable for the first 30 days, in mm; not defined by the source |
| 18 | `30_50DRain (in mm)` | P | Numerical (Continuous) | Rainfall on days 30-50, in mm |
| 19 | `30_50DAI (in mm)` | P | Numerical (Continuous) | "AI" for days 30-50, in mm; undefined |
| 20 | `51_70DRain (in mm)` | P | Numerical (Continuous) | Rainfall on days 51-70, in mm |
| 21 | `51_70AI (in mm)` | P | Numerical (Continuous) | "AI" for days 51-70, in mm; undefined |
| 22 | `71_105DRain (in mm)` | P | Numerical (Continuous) | Rainfall on days 71-105, in mm |
| 23 | `71_105DAI (in mm)` | P | Numerical (Continuous) | "AI" for days 71-105, in mm; undefined |
| 24 | `Min temp_D1_D30` | P | Numerical (Continuous) | Minimum temperature, days 1-30 |
| 25 | `Max temp_D1_D30` | P | Numerical (Continuous) | Maximum temperature, days 1-30 |
| 26 | `Min temp_D31_D60` | P | Numerical (Continuous) | Minimum temperature, days 31-60 |
| 27 | `Max temp_D31_D60` | P | Numerical (Continuous) | Maximum temperature, days 31-60 |
| 28 | `Min temp_D61_D90` | P | Numerical (Continuous) | Minimum temperature, days 61-90 |
| 29 | `Max temp_D61_D90` | P | Numerical (Continuous) | Maximum temperature, days 61-90 |
| 30 | `Min temp_D91_D120` | P | Numerical (Continuous) | Minimum temperature, days 91-120 |
| 31 | `Max temp_D91_D120` | P | Numerical (Continuous) | Maximum temperature, days 91-120 |
| 32 | `Inst Wind Speed_D1_D30 (in Knots)` | P | Numerical | Instantaneous wind speed, days 1-30, in knots |
| 33 | `Inst Wind Speed_D31_D60 (in Knots)` | P | Numerical | Instantaneous wind speed, days 31-60, in knots |
| 34 | `Inst Wind Speed_D61_D90 (in Knots)` | P | Numerical | Instantaneous wind speed, days 61-90, in knots |
| 35 | `Inst Wind Speed_D91_D120 (in Knots)` | P | Numerical | Instantaneous wind speed, days 91-120, in knots |
| 36 | `Wind Direction_D1_D30` | P | Categorical | Wind direction, days 1-30, e.g. SW, NW, ENE, W |
| 37 | `Wind Direction_D31_D60` | P | Categorical | Wind direction, days 31-60 |
| 38 | `Wind Direction_D61_D90` | P | Categorical | Wind direction, days 61-90 |
| 39 | `Wind Direction_D91_D120` | P | Categorical | Wind direction, days 91-120 |
| 40 | `Relative Humidity_D1_D30` | P | Numerical (Continuous) | Relative humidity, days 1-30 |
| 41 | `Relative Humidity_D31_D60` | P | Numerical (Continuous) | Relative humidity, days 31-60 |
| 42 | `Relative Humidity_D61_D90` | P | Numerical (Continuous) | Relative humidity, days 61-90 |
| 43 | `Relative Humidity_D91_D120` | P | Numerical (Continuous) | Relative humidity, days 91-120 |
| 44 | `Trash (in bundles)` | P | Numerical (Integer) | Quantity of "Trash" counted at harvest, in bundles; exact material not specified |
| 45 | `Paddy yield (in Kg)` | P | Numerical (Integer) | Total paddy yield at harvest, in kg |

### 1.2 Problem definition
| Topic | Definition |
|---|---|
| **General problem** | Agricultural agencies collect season records from many farms, but the variety field is often missing or wrong. **Goal:** given a plot's complete season record, identify which rice variety was grown, to fill in or check the variety label. |
| **ML problem** | **Supervised multiclass classification** (3 nominal classes, $K = 3$). Learn $f:\mathbf{x}\rightarrow \hat y$, where $\hat y$ is one of `CO_43`, `delux ponni`, `ponmani`, with **logistic regression**. The multiclass problem is solved with **One-vs-Rest (OvR)**: $K = 3$ independent **binary** logistic-regression models, one per variety, each separating *that variety* (label 1) from *the other two* (label 0). Model $k$ outputs $p_k(\mathbf{x}) = \sigma(\mathbf{w}_k^\top\mathbf{x}+b_k) = \dfrac{1}{1+e^{-(\mathbf{w}_k^\top\mathbf{x}+b_k)}}$, the probability that the plot grew variety $k$, and is fit by minimising its binary cross-entropy $-\frac{1}{n}\sum_i \big[y_{ik}\log p_k(\mathbf{x}_i) + (1-y_{ik})\log(1-p_k(\mathbf{x}_i))\big]$. Prediction: $\hat y = \arg\max_k p_k(\mathbf{x})$, the variety whose model is most confident. The 3 models are trained separately, so their probabilities need not sum to 1 (rescale by $p_k/\sum_j p_j$ if calibrated class probabilities are needed). |
| **Target variable** | `Variety`, nominal with 3 classes and no natural order |
| **Predictors (candidates)** | **All 44 remaining columns**: 37 numeric (incl. `Trash` and `Paddy yield`) + 7 categorical (`Agriblock`, `Soil Types`, `Nursery`, 4 × `Wind Direction`, before encoding). |
| **Sample** | One row = one season's cultivation record for one plot (1-6 ha) in one of 6 agri-blocks, labelled with the single variety grown on it. |
| **Evaluation** | On a held-out test set: **accuracy**, **per-class precision / recall / F1**, **macro-F1** (weights each variety equally, so a smaller class can't be ignored), and the **confusion matrix** |

### 1.3 Assumptions 
| # | Assumption | Why it matters |
|---|---|---|
| A1 | **Prediction time = after harvest.** The record is complete. only the variety label is missing or in doubt | Decides which columns are *available* at prediction time. |
| A2 | Rows are independent plots | Logistic regression's likelihood and K-fold CV both assume independent observations |
| A3 | Weather columns are block-level summaries, not per-plot measurements | Weather can then only separate *blocks*, not varieties *within* a block |
| A4 | `AI` is undocumented; treated as an available but unexplained seasonal variable | Can't include/exclude on documentation alone |
| A5 | Rows are treated as records of real plots. | If true, results describe this dataset and may not generalise to real farms |
| A6 | The 3 labels are correct and exhaustive: every plot grew exactly one of these 3 varieties | The model can only output these 3 classes |

### 1.4 Data-leakage audit — which predictors are usable
**Target leakage** = a predictor that carries information about the target that won't be available at prediction time, inflating test scores while failing in practice.

| Column | Decision | Reason |
|---|---|---|
| `Variety` | target | — |
| `Hectares`, `Agriblock`, `Soil Types`, `Nursery` | candidate | fixed before or at planting |
| `Seedrate`, `LP_Mainfield`, `Nursery area`, `LP_nurseryarea`, `DAP_20days`, `Weed28D_thiobencarb`, `Urea_40Days`, `Potassh_50Days`, `Micronutrients_70Days`, `Pest_60Day` | candidate | in-season management decisions, all recorded before harvest |
| `30DRain … 71_105DRain`, `*AI`, temperature, wind speed/direction, humidity (D1-D120) | candidate | known by the end of the season under A1; block-level only (A3) |
| **`Trash(in bundles)`**, **`Paddy yield(in Kg)`** | candidate | Harvest outcomes. |

**Other leakage**
1. *Pre-processing leakage*: scalers, encoders, PCA and every selection statistic are **fit on training rows only**
2. *Duplicate leakage*: exact duplicate rows are removed **before** the split
3. *Selection leakage*: correlation / ANOVA / MI / chi-square statistics are computed on the **training set only**